In [14]:
import pandas as pd

In [15]:
import numpy as np

In [16]:
df = pd.read_csv('../data/Telco-Customer-Churn-cleaned.csv')

### Step 5.1: Binary Encoding (for Yes/No-style columns)

In [17]:
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
               'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'Churn']

for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0})

# Confirm all 13 columns converted successfully
print(df[binary_cols].dtypes)
print(df[binary_cols].head())

gender              int64
Partner             int64
Dependents          int64
PhoneService        int64
MultipleLines       int64
OnlineSecurity      int64
OnlineBackup        int64
DeviceProtection    int64
TechSupport         int64
StreamingTV         int64
StreamingMovies     int64
PaperlessBilling    int64
Churn               int64
dtype: object
   gender  Partner  Dependents  PhoneService  MultipleLines  OnlineSecurity  \
0       0        1           0             0              0               0   
1       1        0           0             1              0               1   
2       1        0           0             1              0               1   
3       1        0           0             0              0               1   
4       0        0           0             1              0               0   

   OnlineBackup  DeviceProtection  TechSupport  StreamingTV  StreamingMovies  \
0             1                 0            0            0                0   
1            

#### Step 5.2: One-Hot Encoding (for columns with 3+ categories)

In [18]:
multi_cat_cols = ['InternetService', 'Contract', 'PaymentMethod']

df = pd.get_dummies(df, columns=multi_cat_cols, drop_first=True)

print(f"Shape after one-hot encoding: {df.shape}")
print(df.columns.tolist())

Shape after one-hot encoding: (7043, 25)
['Unnamed: 0', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


#### Step 5.3: Resolving the tenure - TotalCharges Redundancy (VIF)

In [19]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# If TotalCharges was already dropped (e.g. cells run out of order),
# pull it fresh from the cleaned CSV just for VIF — don't modify df
if 'TotalCharges' not in df.columns:
    _raw = pd.read_csv('../data/Telco-Customer-Churn-cleaned.csv',
                       usecols=['tenure', 'MonthlyCharges', 'TotalCharges'])
    vif_data = _raw.dropna()
    print('Note: TotalCharges was already dropped; using raw CSV values for VIF display only.')
else:
    vif_data = df[['tenure', 'MonthlyCharges', 'TotalCharges']].dropna()

vif_df = pd.DataFrame()
vif_df['Feature'] = ['tenure', 'MonthlyCharges', 'TotalCharges']
vif_df['VIF'] = [variance_inflation_factor(vif_data[['tenure', 'MonthlyCharges', 'TotalCharges']].values, i)
                 for i in range(3)]
print(vif_df)

          Feature       VIF
0          tenure  6.332328
1  MonthlyCharges  3.355660
2    TotalCharges  8.075070


In [20]:
# Drop TotalCharges only if it still exists (safe to re-run)
if 'TotalCharges' in df.columns:
    df = df.drop('TotalCharges', axis=1)

# Confirm VIF improves after removing TotalCharges
numeric_features_after = ['tenure', 'MonthlyCharges']
vif_data_after = df[numeric_features_after]

vif_df_after = pd.DataFrame()
vif_df_after['Feature'] = vif_data_after.columns
vif_df_after['VIF'] = [variance_inflation_factor(vif_data_after.values, i) for i in range(len(vif_data_after.columns))]
print("VIF after dropping TotalCharges:")
print(vif_df_after)

VIF after dropping TotalCharges:
          Feature       VIF
0          tenure  2.612607
1  MonthlyCharges  2.612607


#### Step 5.5: Final Feature Selection — Which Features Actually Matter?

In [21]:
# Part A: Full correlation with Churn
correlations = df.corr()['Churn'].sort_values(ascending=False)
print(correlations)

Churn                                    1.000000
InternetService_Fiber optic              0.308020
PaymentMethod_Electronic check           0.301919
MonthlyCharges                           0.193356
PaperlessBilling                         0.191825
SeniorCitizen                            0.150889
StreamingTV                              0.063228
StreamingMovies                          0.061382
MultipleLines                            0.040102
PhoneService                             0.011942
Unnamed: 0                               0.010286
gender                                  -0.008612
DeviceProtection                        -0.066160
OnlineBackup                            -0.082255
PaymentMethod_Mailed check              -0.091683
PaymentMethod_Credit card (automatic)   -0.134302
Partner                                 -0.150448
Dependents                              -0.164221
TechSupport                             -0.164674
OnlineSecurity                          -0.171226


In [22]:
# Part B: Random Forest feature importance (quick check)
from sklearn.ensemble import RandomForestClassifier

X = df.drop('Churn', axis=1)
y = df['Churn']

rf_quick = RandomForestClassifier(random_state=42)
rf_quick.fit(X, y)

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_quick.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df)

                                  Feature  Importance
5                                  tenure    0.206366
15                         MonthlyCharges    0.176390
0                              Unnamed: 0    0.160262
16            InternetService_Fiber optic    0.045765
21         PaymentMethod_Electronic check    0.037139
19                      Contract_Two year    0.036155
1                                  gender    0.027745
18                      Contract_One year    0.025623
14                       PaperlessBilling    0.025285
8                          OnlineSecurity    0.025147
11                            TechSupport    0.023759
3                                 Partner    0.023201
9                            OnlineBackup    0.021300
2                           SeniorCitizen    0.020616
7                           MultipleLines    0.019851
10                       DeviceProtection    0.019501
4                              Dependents    0.019377
12                          

In [23]:
# Drop weak features: near-zero in both correlation AND importance
cols_to_drop = [c for c in ['PhoneService', 'gender'] if c in df.columns]
df = df.drop(cols_to_drop, axis=1)

print(f"Shape after feature selection: {df.shape}")
print(df.columns.tolist())

Shape after feature selection: (7043, 22)
['Unnamed: 0', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'Churn', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


#### Step 5.4: Scaling Numeric Features (applied after feature selection)

In [24]:
from sklearn.preprocessing import StandardScaler

# Show before stats
print("BEFORE scaling:")
print(f"  tenure       mean={df['tenure'].mean():.2f}, std={df['tenure'].std():.2f}")
print(f"  MonthlyCharges mean={df['MonthlyCharges'].mean():.2f}, std={df['MonthlyCharges'].std():.2f}")

scaler = StandardScaler()
df[['tenure', 'MonthlyCharges']] = scaler.fit_transform(df[['tenure', 'MonthlyCharges']])

# Show after stats
print("\nAFTER scaling:")
print(f"  tenure       mean={df['tenure'].mean():.2f}, std={df['tenure'].std():.2f}")
print(f"  MonthlyCharges mean={df['MonthlyCharges'].mean():.2f}, std={df['MonthlyCharges'].std():.2f}")

BEFORE scaling:
  tenure       mean=32.37, std=24.56
  MonthlyCharges mean=64.76, std=30.09

AFTER scaling:
  tenure       mean=-0.00, std=1.00
  MonthlyCharges mean=-0.00, std=1.00


In [25]:
# Save final processed dataset
df.to_csv('../data/Telco-Customer-Churn-final.csv', index=False)

print(f"Final dataset saved: {df.shape[0]} rows x {df.shape[1]} columns")
print(df.head())

Final dataset saved: 7043 rows x 22 columns
   Unnamed: 0  SeniorCitizen  Partner  Dependents    tenure  MultipleLines  \
0           0              0        1           0 -1.277445              0   
1           1              0        0           0  0.066327              0   
2           2              0        0           0 -1.236724              0   
3           3              0        0           0  0.514251              0   
4           4              0        0           0 -1.236724              0   

   OnlineSecurity  OnlineBackup  DeviceProtection  TechSupport  ...  \
0               0             1                 0            0  ...   
1               1             0                 1            0  ...   
2               1             1                 0            0  ...   
3               1             0                 1            1  ...   
4               0             0                 0            0  ...   

   PaperlessBilling  MonthlyCharges  Churn  InternetService_